
# Modelagem Dimensional — Camada Gold

## Objetivo

Construir um modelo dimensional em esquema estrela a partir dos registros tratados de interrupções de carga do ONS.

O modelo tem como finalidade disponibilizar dados estruturados para análises históricas, geográficas, operacionais e de impacto das interrupções no sistema elétrico brasileiro.

## Arquitetura

A camada Gold será composta por:

- dim_tempo: atributos temporais.
- dim_localidade: estado e subsistema.
- dim_agente: agente afetado.
- dim_rede: envolvimento da Rede Básica e Rede de Operação.
- fato_interrupcao: medidas e identificadores das interrupções.

## Granularidade

Cada linha da tabela fato representa um registro válido de interrupção de carga proveniente da camada Silver.

Uma perturbação pode estar associada a múltiplos registros. Portanto, o identificador da perturbação não representa a chave primária da tabela fato.

## Estratégia de identificação

Serão utilizadas chaves substitutas determinísticas, geradas a partir dos atributos naturais das dimensões.

A tabela fato será relacionada às dimensões por essas chaves.

## Rastreabilidade

Os identificadores originais do ONS serão preservados. A quantidade de registros da fato deverá corresponder à quantidade de registros válidos da Silver.


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "ons_energia"

TABELA_SILVER = (
    f"{CATALOG}.silver.silver_interrupcao_carga"
)

GOLD = f"{CATALOG}.gold"

df_silver = spark.table(TABELA_SILVER)

print("Registros Silver:", df_silver.count())

assert df_silver.count() == 8309

display(df_silver.limit(5))



## Dimensão Tempo

A dimensão tempo organiza os atributos temporais utilizados nas análises históricas e sazonais.

A chave substituta será construída a partir da data da interrupção, utilizando o formato yyyyMMdd.

A dimensão conterá uma linha por data distinta presente nos registros válidos.

Os atributos derivados incluem ano, mês, trimestre, dia e dia da semana.

O horário original será preservado na tabela fato, permitindo análises por hora sem aumentar desnecessariamente a granularidade da dimensão.


In [0]:

df_dim_tempo = (
    df_silver
    .select("data_interrupcao")
    .distinct()
    .withColumn(
        "sk_tempo",
        F.date_format(
            "data_interrupcao",
            "yyyyMMdd"
        ).cast("int")
    )
    .withColumn(
        "ano",
        F.year("data_interrupcao")
    )
    .withColumn(
        "mes",
        F.month("data_interrupcao")
    )
    .withColumn(
        "trimestre",
        F.quarter("data_interrupcao")
    )
    .withColumn(
        "dia",
        F.dayofmonth("data_interrupcao")
    )
    .withColumn(
        "dia_semana",
        F.dayofweek("data_interrupcao")
    )
    .withColumn(
        "nome_mes",
        F.date_format("data_interrupcao", "MMMM")
    )
    .select(
        "sk_tempo",
        "data_interrupcao",
        "ano",
        "mes",
        "trimestre",
        "dia",
        "dia_semana",
        "nome_mes"
    )
)

display(df_dim_tempo.limit(10))


In [0]:

(
    df_dim_tempo.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.dim_tempo")
)



## Dimensão Localidade

A dimensão localidade representa a distribuição geográfica das interrupções de carga.

Sua granularidade é a combinação entre estado e subsistema elétrico.

A chave substituta será gerada a partir dessa combinação, permitindo preservar as associações presentes na fonte original.

A dimensão possibilita análises por UF e por subsistema, atendendo principalmente à pergunta P2.


In [0]:

df_dim_localidade = (
    df_silver
    .select(
        "id_estado",
        "id_subsistema",
        "nom_subsistema"
    )
    .distinct()
    .withColumn(
        "sk_localidade",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("id_estado"),
                F.col("id_subsistema")
            ),
            256
        )
    )
    .select(
        "sk_localidade",
        "id_estado",
        "id_subsistema",
        "nom_subsistema"
    )
)

display(df_dim_localidade)


In [0]:

df_conflitos_localidade = (
    df_dim_localidade
    .groupBy("sk_localidade")
    .count()
    .filter(F.col("count") > 1)
)

assert df_conflitos_localidade.count() == 0, (
    "Existem descrições conflitantes para a mesma localidade."
)

print("Unicidade da dimensão localidade validada.")


In [0]:

(
    df_dim_localidade.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.dim_localidade")
)



## Dimensão Agente

A dimensão agente identifica as organizações afetadas pelas interrupções registradas pelo ONS.

O nome do agente será utilizado como chave natural, pois a base original não disponibiliza um identificador cadastral exclusivo.

A chave substituta será gerada por hash determinístico.

Essa modelagem permite relacionar os registros de interrupção aos agentes afetados, preservando a denominação informada pela fonte.

A ausência de um identificador cadastral limita a identificação automática de possíveis mudanças de nome ao longo do histórico.


In [0]:

df_dim_agente = (
    df_silver
    .select("nom_agente")
    .distinct()
    .withColumn(
        "sk_agente",
        F.sha2(
            F.col("nom_agente"),
            256
        )
    )
    .select(
        "sk_agente",
        "nom_agente"
    )
)

display(df_dim_agente.limit(10))


In [0]:

(
    df_dim_agente.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.dim_agente")
)



## Dimensão Rede

A dimensão rede classifica os registros conforme os indicadores de envolvimento da Rede Básica e da Rede de Operação do ONS.

Cada combinação dos indicadores S/N constitui um membro da dimensão.

A modelagem permite comparar a magnitude, a duração e a energia não suprida das interrupções conforme o envolvimento das estruturas de rede.

Os indicadores representam características registradas da perturbação e não devem ser interpretados isoladamente como comprovação de causalidade.


In [0]:

df_dim_rede = (
    df_silver
    .select(
        "flg_envolveuredebasica",
        "flg_envolveuredeoperacao"
    )
    .distinct()
    .withColumn(
        "sk_rede",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("flg_envolveuredebasica"),
                F.col("flg_envolveuredeoperacao")
            ),
            256
        )
    )
    .withColumn(
        "envolve_rede_basica",
        F.when(
            F.col("flg_envolveuredebasica") == "S",
            "Sim"
        ).otherwise("Não")
    )
    .withColumn(
        "envolve_rede_operacao",
        F.when(
            F.col("flg_envolveuredeoperacao") == "S",
            "Sim"
        ).otherwise("Não")
    )
)

display(df_dim_rede)


In [0]:

(
    df_dim_rede.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.dim_rede")
)



## Tabela Fato — Interrupções de Carga

A tabela fato concentra as medidas quantitativas das interrupções e as chaves de relacionamento com as dimensões.

### Granularidade

Um registro válido de interrupção de carga da camada Silver.

### Medidas

- Carga interrompida, em MW.
- Tempo médio de recomposição, em minutos.
- Energia não suprida, em MWh.
- Quantidade de registros, com valor unitário igual a 1.

### Regras de agregação

A quantidade de registros é aditiva.

A carga interrompida poderá ser agregada como soma das magnitudes registradas, sem interpretar o resultado como potência simultânea do sistema.

O tempo médio de recomposição não é uma medida aditiva. Sua análise deverá utilizar médias e distribuições.

A energia não suprida será analisada considerando a granularidade da fonte, com verificação de possível repetição entre registros de uma mesma perturbação.


In [0]:

dimensoes = {
    "dim_tempo": "sk_tempo",
    "dim_localidade": "sk_localidade",
    "dim_agente": "sk_agente",
    "dim_rede": "sk_rede"
}

for nome, chave in dimensoes.items():

    df = spark.table(f"{GOLD}.{nome}")

    total = df.count()

    distintos = df.select(chave).distinct().count()

    nulos = df.filter(
        F.col(chave).isNull()
    ).count()

    assert total == distintos, (
        f"Chaves duplicadas em {nome}"
    )

    assert nulos == 0, (
        f"Chaves nulas em {nome}"
    )

    print(
        f"{nome}: {total} registros — OK"
    )


In [0]:

df_fato = (
    df_silver
    .withColumn(
        "sk_tempo",
        F.date_format(
            "data_interrupcao",
            "yyyyMMdd"
        ).cast("int")
    )
    .withColumn(
        "sk_localidade",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("id_estado"),
                F.col("id_subsistema")
            ),
            256
        )
    )
    .withColumn(
        "sk_agente",
        F.sha2(
            F.col("nom_agente"),
            256
        )
    )
    .withColumn(
        "sk_rede",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("flg_envolveuredebasica"),
                F.col("flg_envolveuredeoperacao")
            ),
            256
        )
    )
    .withColumn(
        "qtd_registros",
        F.lit(1)
    )
    .withColumn(
        "sk_interrupcao",
        F.sha2(
            F.to_json(
                F.struct(
                    *[
                        F.col(c)
                        for c in ['cod_perturbacao', 'din_interrupcaocarga', 'id_subsistema', 'nom_subsistema', 'id_estado', 'nom_agente', 'val_cargainterrompida_mw', 'val_tempomedio_minutos', 'val_energianaosuprida_mwh', 'flg_envolveuredebasica', 'flg_envolveuredeoperacao']
                    ]
                )
            ),
            256
        )
    )
)


In [0]:

colunas_negocio = [
    "cod_perturbacao",
    "din_interrupcaocarga",
    "id_subsistema",
    "nom_subsistema",
    "id_estado",
    "nom_agente",
    "val_cargainterrompida_mw",
    "val_tempomedio_minutos",
    "val_energianaosuprida_mwh",
    "flg_envolveuredebasica",
    "flg_envolveuredeoperacao"
]


In [0]:

df_fato = df_fato.select(
    "sk_interrupcao",
    "sk_tempo",
    "sk_localidade",
    "sk_agente",
    "sk_rede",
    "cod_perturbacao",
    "din_interrupcaocarga",
    "val_cargainterrompida_mw",
    "val_tempomedio_minutos",
    "val_energianaosuprida_mwh",
    "qtd_registros"
)

display(df_fato.limit(10))


In [0]:

(
    df_fato.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.fato_interrupcao")
)

print("Tabela fato persistida com sucesso!")



## Validação do modelo dimensional

Após a persistência, serão executados testes de integridade referencial.

Cada chave estrangeira da tabela fato deverá possuir correspondência na respectiva dimensão.

Também será verificada a unicidade da chave da fato e a preservação da quantidade de registros da camada Silver.

A aprovação desses testes demonstra a consistência estrutural do modelo dimensional.


In [0]:

df_fato_validacao = spark.table(
    f"{GOLD}.fato_interrupcao"
)

for nome, chave in dimensoes.items():

    df_dim = spark.table(
        f"{GOLD}.{nome}"
    )

    sem_correspondencia = (
        df_fato_validacao
        .join(
            df_dim.select(chave),
            on=chave,
            how="left_anti"
        )
        .count()
    )

    print(
        f"{nome}: {sem_correspondencia} registros sem correspondência"
    )

    assert sem_correspondencia == 0

print("Integridade referencial validada!")


In [0]:

qtd_silver = df_silver.count()

qtd_fato = df_fato_validacao.count()

qtd_chaves_distintas = (
    df_fato_validacao
    .select("sk_interrupcao")
    .distinct()
    .count()
)

print("Registros Silver:", qtd_silver)
print("Registros Fato:", qtd_fato)
print("Chaves distintas:", qtd_chaves_distintas)

assert qtd_silver == qtd_fato

assert qtd_fato == qtd_chaves_distintas

print("Validação quantitativa concluída!")


In [0]:

comentarios = {
    "dim_tempo":
        "Dimensao temporal das interrupcoes de carga do ONS.",

    "dim_localidade":
        "Dimensao geografica por estado e subsistema eletrico.",

    "dim_agente":
        "Dimensao dos agentes afetados pelas interrupcoes.",

    "dim_rede":
        "Dimensao de envolvimento da Rede Basica e Rede de Operacao.",

    "fato_interrupcao":
        "Tabela fato dos registros validos de interrupcao de carga, com medidas e chaves dimensionais."
}

for tabela, descricao in comentarios.items():

    spark.sql(f"""
        COMMENT ON TABLE {GOLD}.{tabela}
        IS '{descricao}'
    """)

print("Comentários das tabelas registrados.")


In [0]:
%sql

SELECT
    f.cod_perturbacao,
    t.data_interrupcao,
    t.ano,
    t.mes,
    l.id_estado,
    l.nom_subsistema,
    a.nom_agente,
    r.envolve_rede_basica,
    f.val_cargainterrompida_mw,
    f.val_tempomedio_minutos,
    f.val_energianaosuprida_mwh

FROM ons_energia.gold.fato_interrupcao f

INNER JOIN ons_energia.gold.dim_tempo t
    ON f.sk_tempo = t.sk_tempo

INNER JOIN ons_energia.gold.dim_localidade l
    ON f.sk_localidade = l.sk_localidade

INNER JOIN ons_energia.gold.dim_agente a
    ON f.sk_agente = a.sk_agente

INNER JOIN ons_energia.gold.dim_rede r
    ON f.sk_rede = r.sk_rede

LIMIT 20;


## Encerramento da modelagem Gold

O modelo dimensional oficial deste MVP utiliza exclusivamente as tabelas do schema `ons_energia.gold`. As verificações anteriores conferem a integridade referencial, a cardinalidade da tabela fato e a unicidade da chave `sk_interrupcao`. As análises de negócio são executadas no notebook `05_analise_final`, e as verificações adicionais de qualidade, no `04_qualidade_dados`.